# Exploratory Data Auditing: Anomaly Analysis
[Brian C. Keegan, Ph.D.](http://www.brianckeegan.com)
May 2026

Released under an [MIT License](https://opensource.org/licenses/MIT).

This notebook detects continuous, categorical, temporal, spatial, and network
anomalies in U.S. House *Statement of Disbursements* spending. It consumes the
single cleaned artifact produced by [`cleaning.ipynb`](cleaning.ipynb)
(`all_disbursements.csv`, Git-LFS-tracked) and performs **no schema
normalization or datetime parsing of its own** — that is the cleaning
notebook's contract. Analysis-specific transforms (personnel payee-name
normalization, gender/party/office enrichment, per-test statistical
subsetting) remain here by design.

## External inputs

Beyond the cleaned disbursements, the enrichment/reference inputs are produced
by `scripts/fetch_biographical.py` and read from local cached files so the
notebook is reproducible offline — the project's notebook discipline keeps
network retrieval out of the notebook itself:

- `propublica_members.csv` — member id → gender / party / name, built from the
  canonical [congress-legislators](https://unitedstates.github.io/congress-legislators/)
  roster (the legacy ProPublica Congress API this file is named after has been
  retired).
- `census_state.txt`, `census_cenpop2020.csv` — Census ANSI state codes and
  2020 [centers of population](https://www.census.gov/geographies/reference-files/time-series/geo/centers-population.html).
- `member_data_2015_2023.csv` — House Clerk `MemberData.xml` office
  building/room (Wayback Machine 2015–2023, with a live-clerk fallback).

Disbursements provenance:
[U.S. House Statement of Disbursements](https://disbursements.house.gov/),
normalized by `cleaning.ipynb`. Run `python scripts/fetch_biographical.py`
once before executing this notebook.

In [ ]:
import numpy as np
import pandas as pd
pd.options.display.max_columns = 100

idx = pd.IndexSlice

pd.set_option('display.float_format', lambda x: '%.2f' % x)

%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.colors as mcolors
import seaborn as sb

from geopy.distance import geodesic

from scipy import stats
from scipy.spatial.distance import pdist, squareform, cosine


import networkx as nx

In [ ]:
top_cats = [
    'PERSONNEL COMPENSATION',
    'FRANKED MAIL',
    'TRAVEL',
    'RENT COMMUNICATION UTILITIES',
    'PRINTING AND REPRODUCTION',
    'OTHER SERVICES',
    'SUPPLIES AND MATERIALS',
    'EQUIPMENT'
]

## Load data

### Centers of population

In [ ]:
# Census ANSI state codes and 2020 centers of population — cached locally by
# scripts/fetch_biographical.py (canonical: https://www.census.gov/geographies
# /reference-files/time-series/geo/centers-population.html).
state_codes_df = pd.read_csv('census_state.txt', sep='|', dtype={'STATE': str})

cop_df = pd.read_csv('census_cenpop2020.csv', dtype={'STATEFP': str})

cop_df = pd.merge(
    left = cop_df,
    right = state_codes_df,
    left_on = 'STATEFP',
    right_on = 'STATE',
    how = 'outer'
)

cop_df.loc[[52,53,54,55,56],'STATEFP'] = ['60','66','69','74','78']
cop_df.loc[[52,53,54,55,56],'LATITUDE'] = [14.3,13.4,16.4,np.nan,18.2]
cop_df.loc[[52,53,54,55,56],'LONGITUDE'] = [170.7,144.8,145.4,np.nan,-64.6]

cop_df.drop(55,inplace=True)

cop_df['DC_DIST'] = cop_df[['LATITUDE','LONGITUDE']].apply(lambda x: geodesic((x['LATITUDE'],x['LONGITUDE']),(cop_df.loc[8,'LATITUDE'],cop_df.loc[8,'LONGITUDE'])).km,axis=1)

cop_df = cop_df[['STATEFP','STUSAB','STATE_NAME','POPULATION','LATITUDE','LONGITUDE','DC_DIST']]
cop_df

### Disbursements

In [ ]:
# Single cleaned artifact produced by cleaning.ipynb (Git-LFS-tracked). All
# schema/datetime work happened upstream; this notebook only reads it.
all_db_df = pd.read_csv(
    'all_disbursements.csv',
    encoding='utf8',
    parse_dates=['DATE','PERIOD_DATE','START DATE','END DATE'],
    low_memory=False
)

all_db_df.head()

Integration contract with `cleaning.ipynb` — a loud failure here means the cleaned schema drifted. This notebook does no schema or datetime work itself.

In [ ]:
EXPECTED_COLUMNS = [
    'YEAR-QUARTER','YEAR','QUARTER','TERM_QUARTER','BIOGUIDE_ID','OFFICE',
    'PROGRAM','CATEGORY','PAYEE','PURPOSE','AMOUNT','DATE','PERIOD_DATE',
    'DATE_IS_RECONSTRUCTED','START DATE','END DATE','TRANSCODE','RECORDID',
    'VOUCHER_ID']
assert list(all_db_df.columns) == EXPECTED_COLUMNS, all_db_df.columns.tolist()
for _c in ['DATE','PERIOD_DATE','START DATE','END DATE']:
    assert pd.api.types.is_datetime64_any_dtype(all_db_df[_c]), _c
assert all_db_df['YEAR'].min() == 2011, 'cleaned universe must start 2011'
print(f"contract OK: {len(all_db_df):,} rows x "
      f"{len(all_db_df.columns)} cols, "
      f"{all_db_df['YEAR-QUARTER'].nunique()} quarters")

Filter to members' expenses.

Filter to members' expenses. **Decision 3:** the source dropped `BIOGUIDE_ID` from 2023Q1 on, so member-level analyses below cover **2011–2022** by construction; 2023–2025 remain in the cleaned data at office level only.

In [ ]:
members_df = all_db_df[all_db_df['BIOGUIDE_ID'].notnull()]
assert members_df['YEAR'].between(2011, 2022).all(), \
    'members_df must be 2011-2022 (Decision 3)'
members_df['BIOGUIDE_ID'].value_counts()

### Bioguide data

### Pro-Publica

In [ ]:
propublica_members_df = pd.read_csv('propublica_members.csv')
propublica_members_df.head()

In [ ]:
bioguide_gender_party_map = propublica_members_df[['id','gender','party']].drop_duplicates(subset=['id']).set_index('id').to_dict('index')

member_names_map = propublica_members_df.copy()[['id','first_name','last_name']]
member_names_map['full_name'] = member_names_map['first_name'] + ' ' + member_names_map['last_name']
member_names_map = member_names_map.drop_duplicates(subset=['id']).set_index('id')['full_name'].to_dict()

### Member data

### Member office data

The House Clerk `MemberData.xml` snapshots (office building/room, 2015–2023 via
the Wayback Machine with a live-clerk fallback) are retrieved by
`scripts/fetch_biographical.py` and cached as `member_data_2015_2023.csv`,
keeping this notebook free of slow, fragile network loops (notebook
discipline — analysis does no inline data retrieval).

In [ ]:
member_data_df = pd.read_csv('member_data_2015_2023.csv',encoding='utf8')

Extract the office information.

In [ ]:
member_office_df = member_data_df.copy()[['year','bioguideID','office-building','office-room']]

member_office_df['office-floor'] = np.nan

_chob = member_office_df['office-building'] == 'CHOB'
member_office_df.loc[_chob,'office-floor'] = 'CHOB-' + member_office_df.loc[_chob,'office-room'].astype(str).str.get(0)

_lhob = member_office_df['office-building'] == 'LHOB'
member_office_df.loc[_lhob,'office-floor'] = 'LHOB-' + member_office_df.loc[_lhob,'office-room'].astype(str).str.get(1)

_rhob = member_office_df['office-building'] == 'RHOB'
member_office_df.loc[_rhob,'office-floor'] = 'RHOB-' + member_office_df.loc[_rhob,'office-room'].astype(str).str.get(1)

member_office_df.head()


### Join

In [ ]:
members_bioguide_df = pd.merge(
    left = members_df,
    right = propublica_members_df,
    left_on = 'BIOGUIDE_ID',
    right_on = 'id',
    how = 'left'
)

members_bioguide_df.head()

## Continuous anomalies

### Outliers

In [ ]:
_s = members_df.loc[members_df['CATEGORY'] == 'PERSONNEL COMPENSATION','AMOUNT']
counts,bins = np.histogram(_s,bins=25)

f,ax = plt.subplots()
ax.hist(bins[:-1],bins,weights=counts)
ax.set_yscale('log')
ax.set_xlim((-60000,100000))
ax.set_ylim((2e-1,1e6))
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax.set_xlabel('Amount')
ax.set_ylabel('Number of expenses')
ax.set_title('Personnel Compensation, 112th–117th Congresses')

for _i,_bin in enumerate(bins):
    if _bin < 0:
        ax.patches[_i].set_color('tab:red')
        
f.tight_layout()
f.savefig('personnel_compensation.png',dpi=300,bbox_inches='tight')

In [ ]:
len(_s[_s > 50000]), len(_s[_s > 50000]) / len(_s)

In [ ]:
c0 = members_df['CATEGORY'] == 'PERSONNEL COMPENSATION'
c1 = members_df['AMOUNT'] > 50000
members_df.loc[c0 & c1,:].sort_values('AMOUNT',ascending=False)

In [ ]:
c2 = members_df['PAYEE'] == 'VERGHESE MATTHEW M'
members_df.loc[c0 & c1 & c2,:]

### Ending in even digits

HC: Both before and after the decimal

In [ ]:
amounts = members_bioguide_df['AMOUNT'].dropna().apply(np.abs).map('{:.2f}'.format).astype(str)

two_digits_before = amounts.str.split('.').apply(lambda x:str(int(x[0][-2:])))
two_digits_after = amounts.str.split('.').apply(lambda x:x[1])

In [ ]:
two_digits_before_counts = two_digits_before.value_counts()
two_digits_after_counts = two_digits_after.value_counts()

two_digits_before_counts.index = ["{0:02d}".format(int(i)) for i in two_digits_before_counts.index]
two_digits_after_counts.index = ["{0:02d}".format(int(i)) for i in two_digits_after_counts.index]

# last_two_digits_counts = last_two_digits_counts[last_two_digits_counts.index.astype(int) >= 0]

In [ ]:
f,axs = plt.subplots(2,1,figsize=(10,5),subplot_kw={'yscale':'log','ylim':(1e-3,1e0),'ylabel':'Fraction'})

before_frac = two_digits_before_counts / two_digits_before_counts.sum()
after_frac = two_digits_after_counts / two_digits_after_counts.sum()

before_frac.sort_index().plot(width=.67, kind='bar',ax=axs[0])
after_frac.sort_index().plot(width=.67, kind='bar',ax=axs[1])

# axs[0].xaxis.set_major_locator(plt.MaxNLocator(90))
axs[0].set_xticks(range(0,100,10),range(0,100,10),rotation=0)
axs[1].set_xticks(range(0,100,10),range(0,100,10),rotation=0)

axs[0].set_title('Pre-decimal')
axs[1].set_title('Post-decimal')

axs[0].set_xlabel(None)
axs[1].set_xlabel(None)

for _i in range(0,100,10):
    if _i > 0:
        axs[0].patches[_i].set_color('tab:red')
        axs[1].patches[_i].set_color('tab:red')
    
for _i in range(0,100,25):
    if _i > 0:
        axs[0].patches[_i].set_color('tab:orange')
        axs[1].patches[_i].set_color('tab:orange')

f.tight_layout()
f.savefig('pre_post_decimal.png',dpi=300,bbox_inches='tight')

In [ ]:
two_digits_before_counts

### Benford's law

In [ ]:
benfords_d = {}

for i in range(1,10):
    benfords_d[str(i)] = np.log10(1+(1/i))
    
benfords_s = pd.Series(benfords_d)
benfords_s

In [ ]:
def leading_digits_count(s):
    if type(s) == pd.Series:
        _gt0 = s[s > 0]
        _str = _gt0.astype(str)
        _leading = _str.str.get(0)
        _not0 = _leading[_leading != '0']
        
        return _not0.value_counts().sort_index()

#### Aggregate

In [ ]:
leading_digit_dist_by_cat = {}

for _cat in members_df['CATEGORY'].value_counts().iloc[:-3].index:
    _amount = members_df.loc[members_df['CATEGORY'] == _cat,'AMOUNT']
    leading_digit_dist_by_cat[_cat] = leading_digits_count(_amount)
#     leading_digit_dist_by_cat[_cat] = _leading_digits / _leading_digits.sum()

leading_digit_dist_by_cat_df = pd.DataFrame(leading_digit_dist_by_cat)

leading_digit_dist_by_cat_df

In [ ]:
leading_digit_dist_by_cat_norm_df = leading_digit_dist_by_cat_df / leading_digit_dist_by_cat_df.sum()

f,ax = plt.subplots()

leading_digit_dist_by_cat_norm_df.plot(kind='bar',width=.75,ax=ax)
benfords_s.plot(ax=ax,color='grey',label='Benford\'s law',lw=5,zorder=0)
ax.legend()
ax.set_xlabel('Leading digit')
ax.set_ylabel('Proportion')
ax.set_ylim((0,.5))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))

f.tight_layout()
f.savefig('benford_all_cat.png',dpi=300,bbox_inches='tight')

In [ ]:
f,ax = plt.subplots()
leading_digit_dist_by_cat_norm_df['SUPPLIES AND MATERIALS'].plot(lw=5,label='Supplies and Materials',ax=ax)
leading_digit_dist_by_cat_norm_df['PERSONNEL COMPENSATION'].plot(lw=5,label='Personnel Compensation',ax=ax)
benfords_s.plot(ax=ax,lw=5,label='Benford',c='grey')
ax.legend()

In [ ]:
dist_test_by_col = {}

for _col in leading_digit_dist_by_cat_df.columns:
    
    _col_vals = leading_digit_dist_by_cat_df.loc[:,_col]
    _exp_vals = _col_vals.sum() * benfords_s
    
    _ks_test = stats.ks_2samp(_col_vals,_exp_vals)
    _chi2_test = stats.chisquare(_col_vals,_exp_vals)
    
    dist_test_by_col[_col] = {
        'ks_statistic':_ks_test.statistic,'ks_pvalue':_ks_test.pvalue,
        'chi2_statistic':_chi2_test.statistic,'chi2_pvalue':_chi2_test.pvalue
    }
        
dist_test_df = pd.DataFrame(dist_test_by_col).T
dist_test_df.sort_values('chi2_statistic',ascending=False)

In [ ]:
stats.zscore(leading_digit_dist_by_cat_df,axis=1)

#### TODO: Break down bucket(s) by year

#### By representative

In [ ]:
transaction_count = members_df['BIOGUIDE_ID'].value_counts()
members_gt1000_transactions = transaction_count[transaction_count >= 1000].index

In [ ]:
leading_digit_dist_by_member = {}

for _member in members_gt1000_transactions:
    _amount = members_df.loc[members_df['BIOGUIDE_ID'] == _member,'AMOUNT']
    leading_digit_dist_by_member[_member] = leading_digits_count(_amount)
#     leading_digit_dist_by_member[_member] = _leading_digits / _leading_digits.sum()
    
leading_digit_dist_by_member_df = pd.DataFrame(leading_digit_dist_by_member)

leading_digit_dist_by_member_df

In [ ]:
ks_test_by_member = {}
for _member in leading_digit_dist_by_member_df.columns:
    _col_vals = leading_digit_dist_by_member_df[_member]
    _exp_vals = _col_vals.sum() * benfords_s
    
    _ks_test = stats.ks_2samp(_col_vals,_exp_vals)
    _chi2_test = stats.chisquare(_col_vals,_exp_vals)
    
    ks_test_by_member[_member] = {
        'ks_statistic':_ks_test.statistic,'ks_pvalue':_ks_test.pvalue,
        'chi2_statistic':_chi2_test.statistic,'chi2_pvalue':_chi2_test.pvalue
    }
        
dist_test_by_member_df = pd.DataFrame(ks_test_by_member).T
dist_test_by_member_df.sort_values('chi2_statistic',ascending=False).head(20)

#### Member-level travel

In [ ]:
leading_digit_dist_by_member_travel = {}

for _member in members_gt1000_transactions:
    _amount = members_df.loc[(members_df['BIOGUIDE_ID'] == _member) & (members_df['CATEGORY'] == 'TRAVEL'),'AMOUNT']
    leading_digit_dist_by_member_travel[_member] = leading_digits_count(_amount)
#     leading_digit_dist_by_member_travel[_member] = _leading_digits / _leading_digits.sum()
    
leading_digit_dist_by_member_travel_df = pd.DataFrame(leading_digit_dist_by_member_travel)

leading_digit_dist_by_member_travel_df

In [ ]:
ks_test_by_member_travel = {}

for _member in leading_digit_dist_by_member_travel_df.columns:
    _col_vals = leading_digit_dist_by_member_travel_df.loc[:,_member]
    _exp_vals = _col_vals.sum() * benfords_s
    
    _ks_test = stats.ks_2samp(_col_vals,_exp_vals)
    _chi2_test = stats.chisquare(_col_vals,_exp_vals)
    
    ks_test_by_member_travel[_member] = {
        'ks_statistic':_ks_test.statistic,'ks_pvalue':_ks_test.pvalue,
        'chi2_statistic':_chi2_test.statistic,'chi2_pvalue':_chi2_test.pvalue
    }
        
dist_test_by_member_travel_df = pd.DataFrame(ks_test_by_member_travel).T
dist_test_by_member_travel_df.sort_values('chi2_statistic',ascending=False).head(20)

Inspect.

In [ ]:
c0 = members_df['BIOGUIDE_ID'] == 'V000081'
c1 = members_df['CATEGORY'] == 'TRAVEL'

_df = members_df.copy().loc[c0 & c1,:]

_df['leading'] = _df['AMOUNT'].apply(np.abs).astype(str).str.get(0)

_df.head()

In [ ]:
_df['leading'].value_counts(normalize=True)

In [ ]:
_df.loc[_df['leading'] == '6','AMOUNT']

Is there a difference between the median distribution of digits and Benford's law?

In [ ]:
leading_digit_dist_by_member_travel_norm_df = leading_digit_dist_by_member_travel_df / leading_digit_dist_by_member_travel_df.sum()
top10_members_travel = leading_digit_dist_by_member_travel_norm_df.loc[:,dist_test_by_member_travel_df.sort_values('chi2_statistic',ascending=False).head(10).index]
bot10_members_travel = leading_digit_dist_by_member_travel_norm_df.loc[:,dist_test_by_member_travel_df.sort_values('chi2_statistic').head(10).index]

leading_digit_travel_median = leading_digit_dist_by_member_travel_df.median(1)
leading_digit_travel_median_norm = leading_digit_travel_median/leading_digit_travel_median.sum()

stats.chisquare(leading_digit_travel_median,leading_digit_travel_median.sum() * benfords_s)


In [ ]:
f,axs = plt.subplots(2,1,figsize=(8,6),sharey=True,subplot_kw={'ylim':(0,.5)})

top10_members_travel.plot(kind='bar',width=.75,ax=axs[0],legend=False)
axs[0].set_title('Largest deviations')

bot10_members_travel.plot(kind='bar',width=.75,ax=axs[1],legend=False)
axs[1].set_title('Smallest deviations')

for _ax in axs:
    _ax.plot(benfords_s,c='k',lw=3,alpha=.66,label='Benford')
    _ax.plot(leading_digit_travel_median_norm,c='grey',lw=3,alpha=.66,label='Median')
    _ax.legend(loc='center left',bbox_to_anchor=(1,.5))
    _ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    _ax.set_xticklabels(range(1,10), rotation = 0)
    _ax.set_xlabel('Leading digit')

f.suptitle('Deviations in travel expenditures',fontsize=13)

f.tight_layout()

f.savefig('benford_travel_top_bot_10.png',bbox_inches='tight')

#### Member-level personnel expenses

In [ ]:
leading_digit_dist_by_member_personnel = {}

for _member in members_gt1000_transactions:
    _amount = members_df.loc[(members_df['BIOGUIDE_ID'] == _member) & (members_df['CATEGORY'] == 'PERSONNEL COMPENSATION'),'AMOUNT']
    leading_digit_dist_by_member_personnel[_member] = leading_digits_count(_amount)
#     leading_digit_dist_by_member_personnel[_member] = _leading_digits / _leading_digits.sum()
    
leading_digit_dist_by_member_personnel_df = pd.DataFrame(leading_digit_dist_by_member_personnel)

leading_digit_dist_by_member_personnel_df

In [ ]:
ks_test_by_member_personnel = {}

for _member in leading_digit_dist_by_member_personnel_df.columns:
    _col_vals = leading_digit_dist_by_member_personnel_df.loc[:,_member]
    _exp_vals = _col_vals.sum() * benfords_s
    
    _ks_test = stats.ks_2samp(_col_vals,_exp_vals)
    _chi2_test = stats.chisquare(_col_vals,_exp_vals)
    
    ks_test_by_member_personnel[_member] = {
        'ks_statistic':_ks_test.statistic,'ks_pvalue':_ks_test.pvalue,
        'chi2_statistic':_chi2_test.statistic,'chi2_pvalue':_chi2_test.pvalue
    }
        
dist_test_by_member_personnel_df = pd.DataFrame(ks_test_by_member_personnel).T
dist_test_by_member_personnel_df.sort_values('chi2_statistic',ascending=False).head(20)

Is there a difference between the distribution of median leading digits and Benford's law?

In [ ]:
leading_digit_dist_by_member_personnel_norm_df = leading_digit_dist_by_member_personnel_df / leading_digit_dist_by_member_personnel_df.sum()
top10_members_personnel = leading_digit_dist_by_member_personnel_norm_df.loc[:,dist_test_by_member_personnel_df.sort_values('chi2_statistic',ascending=False).head(10).index]
bot10_members_personnel = leading_digit_dist_by_member_personnel_norm_df.loc[:,dist_test_by_member_personnel_df.sort_values('chi2_statistic').head(10).index]

leading_digit_personnel_median = leading_digit_dist_by_member_personnel_df.median(1)
leading_digit_personnel_median_norm = leading_digit_personnel_median/leading_digit_personnel_median.sum()

stats.chisquare(leading_digit_personnel_median,leading_digit_personnel_median.sum() * benfords_s)


In [ ]:
f,axs = plt.subplots(2,1,figsize=(8,6),sharey=True,subplot_kw={'ylim':(0,.6)})

top10_members_personnel.plot(kind='bar',width=.75,ax=axs[0],legend=False)
axs[0].set_title('Largest deviations')

bot10_members_personnel.plot(kind='bar',width=.75,ax=axs[1],legend=False)
axs[1].set_title('Smallest deviations')

for _ax in axs:
    _ax.plot(benfords_s,c='k',lw=3,alpha=.66,label='Benford')
    _ax.plot(leading_digit_personnel_median_norm,c='grey',lw=3,alpha=.66,label='Median')
    _ax.legend(loc='center left',bbox_to_anchor=(1,.5))
    _ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    _ax.set_xticklabels(range(1,10), rotation = 0)
    _ax.set_xlabel('Leading digit')

f.suptitle('Deviations in personnel expenditures',fontsize=13)

f.tight_layout()

f.savefig('benford_personnel_top_bot_10.png',bbox_inches='tight')

Within member, by year, across categories. Biggest member-year-category deviations from Benfords'

In [ ]:
annual_member_cat_leading_digits_df = members_df.groupby(['BIOGUIDE_ID','YEAR','CATEGORY'])['AMOUNT'].apply(leading_digits_count)
annual_member_cat_leading_digits_df.head()

In [ ]:
annual_member_cat_leading_digits_df.unstack([2,1]).sort_index(axis=1)

### Relative size factor

Divide biggest expense by second biggest expense (within member-category-time).

In [ ]:
two_largest_expenses = members_df.groupby(['BIOGUIDE_ID','YEAR','CATEGORY']).agg({'AMOUNT':lambda x:x.nlargest(2)})['AMOUNT']

two_largest_expenses = two_largest_expenses[two_largest_expenses.apply(lambda x:isinstance(x,np.ndarray))]
two_largest_expenses = two_largest_expenses[(two_largest_expenses.apply(lambda x:x[0] > 0)) & (two_largest_expenses.apply(lambda x:x[1] > 0))]

largest_expenses_ratio = two_largest_expenses.apply(lambda x:x[0]/x[1])

In [ ]:
f,ax = plt.subplots()

largest_expenses_ratio.hist(bins=np.logspace(0,4,25),ax=ax)
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(None)
ax.set_ylim((1e0,1e5))
ax.set_xlabel('Ratio')
ax.set_ylabel('Count')
ax.set_title('Ratio between 1st and 2nd largest expenses')

f.tight_layout()
f.savefig('two_largest_ratio.png',bbox_inches='tight')

In [ ]:
len(largest_expenses_ratio[largest_expenses_ratio <= 10]) / len(largest_expenses_ratio)

In [ ]:
largest_expenses_ratio[largest_expenses_ratio > 500].sort_index()

In [ ]:
c0 = members_df['YEAR'] == 2021
c1 = members_df['CATEGORY'] == 'FRANKED MAIL'
c2 = members_df['BIOGUIDE_ID'] == 'P000604'

members_df.loc[c0 & c1 & c2,:].sort_values('AMOUNT',ascending=False)

In [ ]:
# ax = members_df[c1].groupby(['YEAR','BIOGUIDE_ID']).agg({'AMOUNT':'sum'})['AMOUNT'].hist(bins=np.logspace(0,7,25))
ax = members_df.loc[c1,'AMOUNT'].hist(bins=np.logspace(0,5,25))
ax.set_xscale('log')
ax.grid(None)
ax.set_ylim((0,12000))

In [ ]:
ax = members_df[c1].groupby(['YEAR','BIOGUIDE_ID']).agg({'AMOUNT':'nunique'})['AMOUNT'].hist()

In [ ]:
_s = members_df[c1].groupby(['YEAR','BIOGUIDE_ID']).agg({'AMOUNT':'sum'})['AMOUNT']
ax = _s.hist(bins=np.logspace(0,7,25))
ax.set_xscale('log')
ax.grid(None)

In [ ]:
_s.describe()

In [ ]:
members_df[members_df['BIOGUIDE_ID'] == 'P000604']

## Categorical anomalies

In [ ]:
member_expense_count_by_category_df = members_df.groupby(['YEAR-QUARTER','BIOGUIDE_ID'])
member_expense_count_by_category_df = member_expense_count_by_category_df['CATEGORY'].value_counts().unstack(-1).fillna(0)
member_expense_count_by_category_df.drop(columns=['BENEFITS TO FORMER PERSONNEL','PERSONNEL BENEFITS','TRANSPORTATION OF THINGS'],inplace=True)

member_expense_count_by_category_df

In [ ]:
gt10 = member_expense_count_by_category_df.sum(1)
gt10 = gt10[gt10>20]
gt10 = member_expense_count_by_category_df.loc[gt10.index]

_exp = gt10.median(axis=0)
_exp = _exp/_exp.sum()
_df = gt10.div(gt10.sum(1),axis=0)

chisquare_pvalues_expense_counts = pd.Series(data=stats.chisquare(_df,_exp,axis=1).pvalue,index=gt10.index)
gt10.loc[chisquare_pvalues_expense_counts.sort_values().head(20).index]


In [ ]:
_df = member_expense_count_by_category_df.stack().reset_index(level=[2])

axd = plt.figure(figsize=(12,6),layout='constrained').subplot_mosaic(
    """
    ABD
    ACE
    """,
    sharex=True,
)

_order = member_expense_count_by_category_df.median().sort_values().index

sb.barplot(
    data = _df,
    x = 0,
    y = 'CATEGORY',
    order = _order,
    ax = axd['A']
)
axd['A'].set_xlabel("Number of expenses")

sb.barplot(
    data = _df.loc[chisquare_pvalues_expense_counts.sort_values().head(20).index[1]],
    x = 0,
    y = 'CATEGORY',
    order = _order,
    ax = axd['B']
)
axd['B'].yaxis.set_major_locator(mtick.NullLocator())
axd['B'].set_xlabel(None)
axd['B'].set_ylabel(None)
axd['B'].set_title("{1}, {0}".format(*chisquare_pvalues_expense_counts.sort_values().head(20).index[1]))

sb.barplot(
    data = _df.loc[chisquare_pvalues_expense_counts.sort_values().head(20).index[2]],
    x = 0,
    y = 'CATEGORY',
    order = _order,
    ax = axd['C']
)
axd['C'].yaxis.set_major_locator(mtick.NullLocator())
axd['C'].set_xlabel("Number of expenses")
axd['C'].set_ylabel(None)
axd['C'].set_title("{1}, {0}".format(*chisquare_pvalues_expense_counts.sort_values().head(20).index[2]))

sb.barplot(
    data = _df.loc[chisquare_pvalues_expense_counts.sort_values().head(20).index[3]],
    x = 0,
    y = 'CATEGORY',
    order = _order,
    ax = axd['D']
)
axd['D'].yaxis.set_major_locator(mtick.NullLocator())
axd['D'].set_xlabel(None)
axd['D'].set_ylabel(None)
axd['D'].set_title("{1}, {0}".format(*chisquare_pvalues_expense_counts.sort_values().head(20).index[3]))

sb.barplot(
    data = _df.loc[chisquare_pvalues_expense_counts.sort_values().head(20).index[10]],
    x = 0,
    y = 'CATEGORY',
    order = _order,
    ax = axd['E']
)
axd['E'].yaxis.set_major_locator(mtick.NullLocator())
axd['E'].set_xlabel("Number of expenses")
axd['E'].set_ylabel(None)
axd['E'].set_title("{1}, {0}".format(*chisquare_pvalues_expense_counts.sort_values().head(20).index[10]))

plt.tight_layout()
plt.savefig('category_counts.png',dpi=300,bbox_inches='tight')

In [ ]:
c0 = members_df['YEAR-QUARTER'] == '2019Q4'
c1 = members_df['BIOGUIDE_ID'] == 'R000580'
members_df.loc[c0 & c1,:]

In [ ]:
members_df.loc[c0 & c1,'PAYEE'].value_counts()

Compare spending by party.

In [ ]:
member_quarterly_category_amount_df = members_df.groupby(['YEAR-QUARTER','BIOGUIDE_ID','CATEGORY']).agg({'AMOUNT':'sum','OFFICE':len}).reset_index()

member_quarterly_category_amount_df.rename(columns={'OFFICE':'EXPENSE COUNT'},inplace=True)

member_quarterly_category_amount_df = pd.merge(
    left = member_quarterly_category_amount_df,
    right = propublica_members_df[['id','gender','party']].drop_duplicates(subset=['id']),
    left_on = 'BIOGUIDE_ID',
    right_on = 'id',
    how = 'left'
)

member_quarterly_category_amount_df.head()

In [ ]:
c0 = member_quarterly_category_amount_df['EXPENSE COUNT'] >= 5
c1 = ~member_quarterly_category_amount_df['CATEGORY'].isin(['PERSONNEL BENEFITS','TRANSPORTATION OF THINGS'])
c2 = member_quarterly_category_amount_df['party'].isin(['D','R'])

clean_bio_cat_df = member_quarterly_category_amount_df[c0 & c1 & c2]

In [ ]:
member_quarterly_category_amount_df.loc[~member_quarterly_category_amount_df['party'].isin(['D','R']),'id'].unique()

In [ ]:
f,ax = plt.subplots()

sb.barplot(
    data = clean_bio_cat_df,
    x = 'AMOUNT',
    y = 'CATEGORY',
    hue = 'gender',
    width = .8,
    palette = ['tab:orange','tab:green'],
    ax = ax
)

ax.set_ylabel(None)
ax.set_xscale('log')
ax.set_xlim((1e3,1e6))

for i,_cat in enumerate(clean_bio_cat_df['CATEGORY'].unique()):
    _c0 = clean_bio_cat_df['CATEGORY'] == _cat
    _c_men = clean_bio_cat_df['gender'] == 'M'
    _c_women = clean_bio_cat_df['gender'] == 'F'
    
    _men = clean_bio_cat_df.loc[_c0 & _c_men,'AMOUNT']
    _women = clean_bio_cat_df.loc[_c0 & _c_women,'AMOUNT']
    
    _ttest = stats.ttest_ind(_men,_women)
    
    print("{0}: {1:.3f}.\nMen {2:.0f}±{3:.0f}\nWomen {4:.0f}±{5:.0f}\n".format(_cat,_ttest.pvalue,_men.mean(),_men.std(),_women.mean(),_women.std()))
    
    _mult = 1.33
    _max = np.max([_men.mean(),_women.mean()])
    
    if _ttest.pvalue < 0.001:
        ax.annotate('***',(_max.mean() * _mult, i),fontsize=12,ha='center',va='center')
    elif _ttest.pvalue < 0.01:
        ax.annotate('**',(_max.mean() * _mult, i + .25),fontsize=12,ha='center',va='center')
    elif _ttest.pvalue < 0.05:
        ax.annotate('*',(_max.mean() * _mult, i + .25),fontsize=12,ha='center',va='center')
        
f.tight_layout()
f.savefig('category_spending_gender.png',dpi=300,bbox_inches='tight')

In [ ]:
f,ax = plt.subplots()

sb.barplot(
    data = clean_bio_cat_df,
    x = 'AMOUNT',
    y = 'CATEGORY',
    hue = 'party',
    width = .8,
    palette = ['blue','red'],
    ax = ax
)

ax.set_ylabel(None)
ax.set_xscale('log')
ax.set_xlim((1e3,1e6))

for i,_cat in enumerate(clean_bio_cat_df['CATEGORY'].unique()):
    _c0 = clean_bio_cat_df['CATEGORY'] == _cat
    _c_dem = clean_bio_cat_df['party'] == 'D'
    _c_rep = clean_bio_cat_df['party'] == 'R'
    
    _dem = clean_bio_cat_df.loc[_c0 & _c_dem,'AMOUNT']
    _rep = clean_bio_cat_df.loc[_c0 & _c_rep,'AMOUNT']
    
    _ttest = stats.ttest_ind(_dem,_rep)
    
    print("{0}: {1:.2}".format(_cat,_ttest.pvalue))
    
    _mult = 1.33
    _max = np.max([_dem.mean(),_rep.mean()])
    
    if _ttest.pvalue < 0.001:
        ax.annotate('***',(_max.mean() * _mult, i),fontsize=12,ha='center',va='center')
    elif _ttest.pvalue < 0.01:
        ax.annotate('**',(_max.mean() * _mult, i + .25),fontsize=12,ha='center',va='center')
    elif _ttest.pvalue < 0.05:
        ax.annotate('*',(_max.mean() * _mult, i + .25),fontsize=12,ha='center',va='center')
        
f.tight_layout()
f.savefig('category_spending_party.png',dpi=300,bbox_inches='tight')

HC:
1. Pull list of members (systems controls) who "left/lost access/separated" and still getting compensated/reimbursed. Are payments acceptable? Cross-over between temporal and categorical anomalies.
2. Laurie website on fraud scores for members of congress and we validate against these results. External validity for our ensemble of scores.

## Temporal anomalies

In [ ]:
members_travel_df = members_df[members_df['CATEGORY'] == 'TRAVEL']
members_travel_df.head()

#### Check for pre-billing

"END DATE" - "DATE" if negative: Travel advances vs. travel reimbursements

In [ ]:
before_dates = members_travel_df['DATE'] - members_travel_df['START DATE']
before_dates = before_dates/pd.Timedelta(1,'d')
before_dates = before_dates[before_dates < 0]

In [ ]:
members_travel_df.loc[before_dates.sort_values().index,'YEAR'].value_counts().sort_index()

#### Check for impact of COVID-19

In [ ]:
ax = members_travel_df.groupby('YEAR-QUARTER').agg({'AMOUNT':'sum'})['AMOUNT'].plot(lw=3)
ax.axvline(37,color='r')


In [ ]:
_df0 = members_travel_df[members_travel_df['PURPOSE'].isin(['COMMERCIAL TRANSPORTATION','AIRFARE COMMERCIAL TRANSPORT'])]
_df1 = members_travel_df[members_travel_df['PURPOSE'].isin(['PRIVATE AUTO MILEAGE','GASOLINE','TAXI/PARKING/TOLLS'])]
_df2 = members_travel_df[members_travel_df['PURPOSE'].isin(['LODGING','MEALS'])]

f,ax = plt.subplots()
_df0_agg = _df0.groupby('YEAR-QUARTER').agg({'AMOUNT':'sum'})['AMOUNT']
_df1_agg = _df1.groupby('YEAR-QUARTER').agg({'AMOUNT':'sum'})['AMOUNT']
_df2_agg = _df2.groupby('YEAR-QUARTER').agg({'AMOUNT':'sum'})['AMOUNT']

_df0_norm = _df0_agg.div(_df0_agg.loc[['2019Q1','2019Q2','2019Q3','2019Q4']].mean())
_df1_norm = _df1_agg.div(_df1_agg.loc[['2019Q1','2019Q2','2019Q3','2019Q4']].mean())
_df2_norm = _df2_agg.div(_df2_agg.loc[['2019Q1','2019Q2','2019Q3','2019Q4']].mean())

_df0_norm.plot(lw=3,ax=ax,label='Commercial travel')
_df1_norm.plot(lw=3,ax=ax,label='Car travel')
_df2_norm.plot(lw=3,ax=ax,label='Lodging & meals')

ax.axvline(37,color='tab:red',ls='--',lw=1)
ax.legend()
ax.set_ylim((0,2))
ax.set_ylabel('Normalized expenditures')
ax.set_xlabel(None)
ax.annotate(
    'COVID-19 shutdowns',
    xy=(37,1.2),
    xytext=(21,1.4),
    color='tab:red',
    arrowprops={
        'facecolor':'tab:red',
        'edgecolor':'tab:red',
        'arrowstyle':'simple',
        'connectionstyle':'arc3,rad=-0.25'
    }
)

f.tight_layout()
f.savefig('temporal_covid_shutdowns.png',dpi=300,bbox_inches='tight')

In [ ]:
_df1_agg.iloc[36:48].shape,_df1_agg.iloc[24:36].shape

In [ ]:
stats.ttest_ind(_df1_agg.iloc[36:48],_df1_agg.iloc[24:36])

HC: Break it on party affiliation to see how it changes.

In [ ]:
travel_by_member = members_travel_df.groupby(['YEAR-QUARTER','BIOGUIDE_ID']).agg({'AMOUNT':'sum'}).reset_index()

travel_by_member[['gender','party']] = travel_by_member['BIOGUIDE_ID'].apply(lambda x:pd.Series(bioguide_gender_party_map.get(x)))

travel_by_member.head()


In [ ]:
travel_by_member.groupby(['YEAR-QUARTER','party']).agg({'AMOUNT':'sum','BIOGUIDE_ID':'nunique'})

In [ ]:
travel_gender_df = travel_by_member.groupby(['YEAR-QUARTER','gender']).agg({'AMOUNT':'sum','BIOGUIDE_ID':'nunique'})
travel_gender_df = travel_gender_df['AMOUNT'] / travel_gender_df['BIOGUIDE_ID']
travel_gender_df = travel_gender_df.unstack(1)
travel_gender_df = travel_gender_df.div(travel_gender_df.loc[['2019Q1','2019Q2','2019Q3','2019Q4']].mean())

travel_party_df = travel_by_member.groupby(['YEAR-QUARTER','party']).agg({'AMOUNT':'sum','BIOGUIDE_ID':'nunique'})
travel_party_df = travel_party_df['AMOUNT'] / travel_party_df['BIOGUIDE_ID']
travel_party_df = travel_party_df.unstack(1)
travel_party_df = travel_party_df.div(travel_party_df.loc[['2019Q1','2019Q2','2019Q3','2019Q4']].mean())

f,axs = plt.subplots(2,1,sharex=True)
travel_gender_df.loc[:,['M','F']].plot.line(ax=axs[0],lw=3,color=['tab:orange','tab:green'])
travel_party_df.loc[:,['D','R']].plot(ax=axs[1],lw=3,color=['blue','red'])

axs[0].set_ylim((0,2))
axs[1].set_ylim((0,2))

axs[0].legend(['Male','Female'],loc='upper left')
axs[1].legend(['Democratic','Republican'],loc='upper left')

axs[0].set_ylabel('Normalized expenditures')
axs[1].set_ylabel('Normalized expenditures')
axs[1].set_xlabel(None)

axs[0].axhline(1,ls='--',lw=.5,c='k')
axs[1].axhline(1,ls='--',lw=.5,c='k')

axs[0].axvline(37,color='tab:red',ls='--',lw=1)
axs[1].axvline(37,color='tab:red',ls='--',lw=1)

f.tight_layout()

In [ ]:
travel_by_member_pivot = travel_by_member.pivot(index=['BIOGUIDE_ID'],columns=['YEAR-QUARTER'],values='AMOUNT')

pandemic_impact_norm = travel_by_member_pivot.loc[:,'2020Q2'] / travel_by_member_pivot.loc[:,['2019Q1','2019Q2','2019Q3','2019Q4']].mean(1)
pandemic_impact_norm.dropna(inplace=True)

f,ax = plt.subplots()
pandemic_impact_norm.hist(bins=np.linspace(0,2,25),ax=ax)
ax.grid(None)
ax.set_xlabel('Ratio')
ax.set_ylabel('Count')

f.tight_layout()
f.savefig('pandemic_impact_hist.png',dpi=300,bbox_inches='tight')

In [ ]:
len(pandemic_impact_norm[pandemic_impact_norm < 1]) / len(pandemic_impact_norm)

In [ ]:
pandemic_impact_norm[pandemic_impact_norm >= 1]

In [ ]:
pandemic_impact_norm.mean()

In [ ]:
travel_by_member_pivot.loc['H001065',['2019Q1','2019Q2','2019Q3','2019Q4']].mean(), travel_by_member_pivot.loc['H001065','2020Q2']


In [ ]:
c0 = members_travel_df['BIOGUIDE_ID'] == 'H001065'
c1 = members_travel_df['YEAR-QUARTER'] == '2020Q2'
c2 = members_travel_df['CATEGORY'] == 'TRAVEL'
c3 = members_travel_df['PURPOSE'] == 'CAR RENTAL'
members_travel_df.loc[c0 & c1 & c2 & c3,:]#'AMOUNT'].sum()

In [ ]:
members_travel_df.loc[c0 & c1 & c2,:].groupby('PURPOSE').agg({'AMOUNT':'sum'})

In [ ]:
c0 = members_travel_df['BIOGUIDE_ID'] == 'H001065'
c1 = members_travel_df['YEAR-QUARTER'] == '2020Q2'
c2 = members_travel_df['CATEGORY'] == 'TRAVEL'
members_travel_df.loc[c0 & c1 & c2,:].groupby('PURPOSE').agg({'AMOUNT':'sum'})

In [ ]:
c0 = members_travel_df['BIOGUIDE_ID'] == 'H001065'
c4 = members_travel_df['YEAR-QUARTER'] == '2019Q2'
c2 = members_travel_df['CATEGORY'] == 'TRAVEL'
members_travel_df.loc[c0 & c4 & c2,:].groupby('PURPOSE').agg({'AMOUNT':'sum'})

In [ ]:
members_travel_df.loc[c0 & c1 & c2,:].groupby('PURPOSE').agg({'AMOUNT':'sum'}).loc['CAR RENTAL']/members_travel_df.loc[c1 & c2 & c3,:].groupby('BIOGUIDE_ID').agg({'AMOUNT':'sum'}).mean()

#### Cyclical spending

In [ ]:
populated_categories = members_df['CATEGORY'].value_counts().index[:-3]
member_quarterly_spending_by_category_df = members_df[members_df['CATEGORY'].isin(populated_categories)]
member_quarterly_spending_by_category_df = member_quarterly_spending_by_category_df.groupby(['BIOGUIDE_ID','YEAR','QUARTER','CATEGORY']).agg({'AMOUNT':'sum'})
member_quarterly_spending_by_category_df.reset_index(inplace=True)

f,ax = plt.subplots(figsize=(6,8))
sb.barplot(
    data = member_quarterly_spending_by_category_df,
    x = 'AMOUNT',
    y = 'CATEGORY',
    hue = 'QUARTER',
    ax = ax
)

ax.legend(loc='center left',bbox_to_anchor=(1,.5))
ax.set_xscale('log')
ax.set_xlim((1e3,1e6))

In [ ]:
populated_categories = members_df['CATEGORY'].value_counts().index[:-3]
member_termly_spending_by_category_df = members_df[members_df['CATEGORY'].isin(populated_categories)]
member_termly_spending_by_category_df = member_termly_spending_by_category_df.groupby(['BIOGUIDE_ID','YEAR','TERM_QUARTER','CATEGORY']).agg({'AMOUNT':'sum'})
member_termly_spending_by_category_df.reset_index(inplace=True)

f,ax = plt.subplots()
sb.barplot(
    data = member_termly_spending_by_category_df,
    x = 'AMOUNT',
    y = 'CATEGORY',
    hue = 'TERM_QUARTER',
    order = sorted(member_termly_spending_by_category_df['CATEGORY'].unique()),
    ax = ax
)

ax.legend(title='Term quarter',loc='lower right')#,bbox_to_anchor=(1,.5))
ax.set_xscale('log')
ax.set_xlim((1e2,1e6))
ax.set_ylabel(None)

f.tight_layout()
f.savefig('spending_term_quarter.png',dpi=300,bbox_inches='tight')


Q3 AY is Q4 FY, members are spending down budgets in Q3 before appropriations.

HC: Comparing newcomer to incumbent, HC to find cite for govt expenditures increase in FY Q4

In [ ]:
quarterly_category_spending = pd.pivot_table(
    data = member_quarterly_spending_by_category_df,
    index = 'CATEGORY',
    columns = 'QUARTER',
    values = 'AMOUNT',
    aggfunc = 'median'
)

quarterly_category_spending

In [ ]:
for _cat in quarterly_category_spending.index:
    _obs = quarterly_category_spending.loc[_cat]
    _exp = [quarterly_category_spending.loc[_cat].mean()] * 4
    _chi2 = stats.chisquare(_obs,_exp)
    print(_cat,_chi2.pvalue)

In [ ]:
quarterly_category_spending = pd.pivot_table(
    data = member_termly_spending_by_category_df,
    index = 'CATEGORY',
    columns = 'TERM_QUARTER',
    values = 'AMOUNT',
    aggfunc = 'median'
)

quarterly_category_spending

#### Interevent timing 

HC: Pre/post presidential election, do franked mail costs change in month before and after election?  
HC: Conscienciousness about filing expenses, auditing for good

In [ ]:
interevent_data = members_df.copy().sort_values(['BIOGUIDE_ID','CATEGORY','DATE'],ascending=True)
interevent_data = interevent_data.groupby(['BIOGUIDE_ID','CATEGORY'])
interevent_data = interevent_data['DATE'].diff()/pd.Timedelta(1,'d')
interevent_data.dropna(inplace=True)

interevent_counts = interevent_data.value_counts()
interevent_counts.index.name = 'index'
interevent_counts.sort_index(inplace=True)

In [ ]:
f,ax = plt.subplots()

interevent_data.hist(bins=np.logspace(0,3,50),grid=False,ax=ax)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Days between expenses')
ax.set_ylabel('Count')
ax.set_ylim((1e0,1e6))

ax.axvline(7,c='tab:orange',label='One week')
ax.axvline(14,c='tab:red',label='Two weeks')
ax.axvline(30,c='tab:green',label='One month')
ax.axvline(90,c='tab:purple',label='One quarter')
ax.axvline(365*2,c='tab:brown',label='Two years')

ax.legend(fontsize=8)

f.tight_layout()
f.savefig('interevent_anomalies.png',dpi=300,bbox_inches='tight')

In [ ]:
interevent_data.value_counts(normalize=1).sort_index().head(10)

In [ ]:
interevent_data.value_counts().sort_index().reset_index()

In [ ]:
_df = interevent_data.value_counts().sort_index().reset_index()

ax = _df.plot(kind='scatter',x='DATE',y='count')
ax.set_xscale('log')
ax.set_yscale('log')

ax.axvline(7,c='tab:orange',label='One week')
ax.axvline(14,c='tab:red',label='Two weeks')
ax.axvline(30,c='tab:green',label='One month')
ax.axvline(90,c='tab:purple',label='One quarter')
ax.axvline(365*2,c='tab:brown',label='Two years')

HC: People at the 2-year bump are likely to have more discrepancies, they're loading up all their expenses at the end  
HC: Patterns hold up across categories? 90-day rule for reimbursements being taxable forcing stuff  
HC: Did rule change on reimbursements and how is that affecting behavior?  

## Spatial anomalies

HC: Compare members within state to each other, particularly for travel, Guam travel should be highest  
HC: Travel receipts for times when they were also voting?  
BK: Spatial patterns by office? Social influence/comparison could drive anomaly-generating behavior  

In [ ]:
members_bioguide_travel_df = members_bioguide_df[members_bioguide_df['CATEGORY'] == 'TRAVEL']
members_bioguide_travel_df

In [ ]:
annual_member_travel = members_bioguide_travel_df.groupby(['state','YEAR']).agg({'AMOUNT':'sum','BIOGUIDE_ID':'nunique'})
annual_member_travel['per_member'] = annual_member_travel['AMOUNT'] / annual_member_travel['BIOGUIDE_ID']
annual_member_travel_per_member = annual_member_travel['per_member'].unstack('YEAR')
annual_member_travel_per_member.head()

In [ ]:
# avg_travel_spending = annual_member_travel_per_member.mean(axis=1).reset_index()
avg_travel_spending = annual_member_travel_per_member.stack().reset_index()
avg_travel_spending.columns = ['state','year','amount']

_merged = pd.merge(
    left = cop_df,
    right = avg_travel_spending,
    left_on = 'STUSAB',
    right_on = 'state',
    how = 'outer'
)

_merged = _merged[_merged['DC_DIST'] > 0]

f,ax = plt.subplots()
sb.regplot(data=_merged,x='DC_DIST',y='amount',line_kws={'color':'k'},scatter_kws={'s':13},ax=ax)
ax.set_xscale('log')
ax.set_xlim((1e1,2e4))
ax.set_xlabel('Distance from D.C. (km)')
ax.set_ylabel('Per-member travel spending')
ax.set_ylim((0,1.6e6))
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

f.tight_layout()
f.savefig('travel_distance.png',dpi=300,bbox_inches='tight')

HC: Pull out the members with the largest deviation/residuals from the trend

In [ ]:
dist_travel_reg = stats.linregress(_merged['DC_DIST'],_merged['amount'])
dist_travel_reg

In [ ]:
dist_travel_reg.rvalue**2

In [ ]:
state_yhats = {}
for _state,_dist in cop_df[['STATE_NAME','DC_DIST']].values:
    state_yhats[_state] = dist_travel_reg.slope * _dist + dist_travel_reg.intercept
    
travel_dist_yhat_df = pd.DataFrame(data=state_yhats.values(),index=state_yhats.keys(),columns=['yhat'])

travel_dist_yhat_merged = pd.merge(
    left = _merged,
    right = travel_dist_yhat_df,
    left_on = 'STATE_NAME',
    right_index = True,
    how = 'left'
)

travel_dist_yhat_merged['residual'] = travel_dist_yhat_merged['amount'] - travel_dist_yhat_merged['yhat']
travel_dist_yhat_merged['residual_pct'] = travel_dist_yhat_merged['residual'] / travel_dist_yhat_merged['amount']

In [ ]:
travel_dist_yhat_merged.loc[travel_dist_yhat_merged['residual'].abs().nlargest(20).index]

In [ ]:
travel_dist_yhat_merged.loc[travel_dist_yhat_merged['residual_pct'].abs().nlargest(20).index,'STATE_NAME'].value_counts()

In [ ]:
travel_dist_yhat_merged.loc[travel_dist_yhat_merged['residual_pct'].abs().nlargest(20).index,:]

In [ ]:
travel_dist_yhat_merged.loc[travel_dist_yhat_merged['residual_pct'].nlargest(20).index,:]

#### Office building proximity

In [ ]:
annual_category_spending = pd.pivot_table(
    data = members_df,
    index = ['YEAR','BIOGUIDE_ID'],
    columns = ['CATEGORY'],
    values = 'AMOUNT',
    aggfunc = 'sum'
)

# Filter to relevant years and columns
annual_category_spending = annual_category_spending.loc[2015:,top_cats]

# Calculate z-scores for spending in each category
# annual_category_spending_zscore = annual_category_spending.groupby(level=0).apply(stats.zscore,nan_policy='omit')
# annual_category_spending_zscore.index = annual_category_spending_zscore.index.droplevel(0)

annual_category_spending_office = pd.merge(
    left = annual_category_spending.reset_index(),
    right = member_office_df,
    left_on = ['BIOGUIDE_ID','YEAR'],
    right_on = ['bioguideID','year'],
    how = 'inner'
)

annual_category_spending_office['YEAR'] = annual_category_spending_office['YEAR'].astype('Int64')

annual_category_spending_office.head()

In [ ]:
_df = annual_category_spending_zscore_office.set_index(['YEAR','BIOGUIDE_ID']).loc[:,top_cats].dropna(how='all')
_df

In [ ]:
within_floor_std = annual_category_spending_office.groupby(['YEAR','office-floor']).agg({i:np.std for i in top_cats})

within_building_std = annual_category_spending_office.groupby(['YEAR','office-building']).agg({i:np.std for i in top_cats})

within_year_std = annual_category_spending_office.groupby(['YEAR']).agg({i:np.std for i in top_cats})


In [ ]:
_df = within_floor_std.loc[2015].apply(stats.zscore)

pw_df = pd.DataFrame(
    data = 1 - squareform(pdist(_df,'cosine')),
    columns = _df.index,
    index = _df.index
)

sb.heatmap(pw_df)

## Textual anomalies

HC: Benford's for words  
HC: Variance in payee, purpose, etc. by category or is it drop-down  
HC: Consistency in purposes within categories - confirm that some purposes only belong to specific categories/payees. Find a good, bad, and ugly  
HC: Dictionary-based approach of unexpected or undesireable words found in corpus  
HC: Fuzzy matching  

People showing up as payee under supplies? "GRONEMAN BELINDA M." as "OFFICE SUPPLIES (OUTSIDE)"

In [ ]:
members_df.loc[members_df['CATEGORY'] == 'SUPPLIES AND MATERIALS']#,'PURPOSE'].value_counts()

In [ ]:
members_df[members_df['PAYEE'].fillna('').str.lower().str.contains('disney')]

In [ ]:
members_df[members_df['PAYEE'].fillna('').str.lower().str.contains('golf')]

In [ ]:
ax = pd.Series(members_df['PURPOSE'].value_counts().values).hist(bins=np.logspace(0,6,25))
ax.set_xscale('log')
ax.set_yscale('log')

In [ ]:
members_df[members_df['PAYEE'].fillna('').str.contains('GRONEMAN')]

In [ ]:
members_df.loc[members_df['PAYEE'].fillna('').str.contains('GRONEMAN'),'PAYEE'].unique()

## Relational anomalies

HC: More than one anomaly across tests, are there anomalies robust across types of tests?  
BK: Aides working for multiple offices in a single year?   
HC: Relationships within data + relationships among anomalies  

In [ ]:
personnel_compensation_df

In [ ]:
personnel_compensation_df = members_df[members_df['CATEGORY'] == 'PERSONNEL COMPENSATION']

annual_office_personnel_df = personnel_compensation_df.groupby(['YEAR','BIOGUIDE_ID','PAYEE']).agg({'AMOUNT':'sum'})
annual_office_personnel_df.reset_index(inplace=True)
annual_office_personnel_df.dropna(subset=['PAYEE'],inplace=True)
annual_office_personnel_df.head()

Try to clean up PAYEE names.

In [ ]:
yq_total_personnel = personnel_compensation_df.groupby(['YEAR-QUARTER','BIOGUIDE_ID','PAYEE']).agg({'AMOUNT':'sum'})
yq_total_personnel.reset_index(inplace=True)
yq_total_personnel['PAYEE_CLEANED'] = np.nan

In [ ]:
def name_reorganizer(s):
    _split_l = s.strip().split(' ')
    _len = len(_split_l)
    if _len == 1:
        return _split_l
    elif _len == 2:
        return _split_l[-1] + ' ' + _split_l[0]
    elif _len == 3:
        return _split_l[-2] + ' ' + _split_l[-1] + ' ' + _split_l[0]
    elif _len == 4:
        return _split_l[-2] + ' ' + _split_l[-1] + ' ' + _split_l[0] + ' ' + _split_l[1]
    elif _len == 5:
        return _split_l[-2] + ' ' + _split_l[-1] + ' ' + _split_l[0] + ' ' + _split_l[1] + ' ' + _split_l[2]

In [ ]:
yq_total_personnel.loc[name_array_len_sep_comma[name_array_len_sep_comma != 2],:]

In [ ]:
_s = yq_total_personnel.groupby('YEAR-QUARTER').agg({'PAYEE':lambda x:x.str.contains(',').sum()})['PAYEE']
comma_in_name_quarters = _s[_s > 0].index

# Filter to quarters where there are commas present
pre_2016Q4 = yq_total_personnel['YEAR-QUARTER'].isin(comma_in_name_quarters)

# Make a series that separates PAYEE on commas and counts the number of name elements -- ideally 2
name_array_len_sep_comma = yq_total_personnel.loc[pre_2016Q4,'PAYEE'].str.split(',').apply(len)

# Re-order names with exactly two name elements and additional cleanup
exactly_2 = name_array_len_sep_comma[name_array_len_sep_comma == 2]
pre_2016Q4_cleaned_names = yq_total_personnel.loc[exactly_2.index,'PAYEE'].str.split(',').apply(lambda x:x[1] + ' ' + x[0])
pre_2016Q4_cleaned_names = pre_2016Q4_cleaned_names.str.replace('  ', ' ').str.strip()

# Filter to quarters where there are no commas present
post_2016Q4 = ~yq_total_personnel['YEAR-QUARTER'].isin(comma_in_name_quarters)
post_2016Q4_cleaned_names = yq_total_personnel.loc[post_2016Q4,'PAYEE']
post_2016Q4_cleaned_names = post_2016Q4_cleaned_names.str.replace('  ', ' ').str.strip()
post_2016Q4_cleaned_names = post_2016Q4_cleaned_names.apply(name_reorganizer)

# Assign back to a new column
yq_total_personnel.loc[pre_2016Q4_cleaned_names.index,'PAYEE_CLEANED'] = pre_2016Q4_cleaned_names
yq_total_personnel.loc[post_2016Q4_cleaned_names.index,'PAYEE_CLEANED'] = post_2016Q4_cleaned_names

# Remove periods from names
yq_total_personnel.loc[:,'PAYEE_CLEANED'] = yq_total_personnel.loc[:,'PAYEE_CLEANED'].str.replace('.','').fillna('')
yq_total_personnel['PAYEE_CLEANED'] = yq_total_personnel['PAYEE_CLEANED'].replace({'T E ANFINSON':'THOMAS E ANFINSON'})

In [ ]:
yq_total_personnel_filtered = yq_total_personnel[yq_total_personnel['AMOUNT'] >= 1000]

_agg = {'AMOUNT':'sum'}
personnel_el_df = yq_total_personnel_filtered.groupby(['YEAR-QUARTER','BIOGUIDE_ID','PAYEE_CLEANED']).agg(_agg)
# personnel_el_df.index.names = ['yearquarter','id','payee']
personnel_el_df.reset_index(inplace=True)
personnel_el_df['member'] = personnel_el_df['BIOGUIDE_ID'].map(member_names_map).str.upper()
personnel_el_df.columns = ['yearquarter','id','payee','amount','member']

personnel_el_df[['gender','party']] = personnel_el_df['id'].apply(lambda x:pd.Series(bioguide_gender_party_map.get(x)))
personnel_el_df.dropna(subset=['member'],inplace=True)

filtered_personnel_el = personnel_el_df[personnel_el_df['party'].isin(['D','R'])]

In [ ]:
personnel_bp_g = nx.from_pandas_edgelist(
    df = filtered_personnel_el,
    source = 'member',
    target = 'payee',
    edge_attr = ['amount']
)

nx.write_gexf(personnel_bp_g,'personnel_bp.gexf')

personnel_bp_g.number_of_nodes(), personnel_bp_g.number_of_edges()

In [ ]:
set(personnel_el_df['member']) & set(personnel_el_df['payee'])

In [ ]:
personnel_el_df[~personnel_el_df['party'].isin(['D','R','I'])]

In [ ]:
personnel_attrs = filtered_personnel_el.groupby('payee').agg({'party':'unique','gender':'unique'})
personnel_attrs['party'] = personnel_attrs['party'].fillna('').apply(sorted).str.join(',')
personnel_attrs['gender'] = personnel_attrs['gender'].fillna('').apply(sorted).str.join(',')
personnel_attrs['joint'] = personnel_attrs['party'] + ',' + personnel_attrs['gender']

personnel_attrs['joint'].value_counts()

In [ ]:
personnel_proj_g = nx.bipartite.projection.weighted_projected_graph(personnel_bp_g,filtered_personnel_el['payee'].unique())

joint_d = personnel_attrs['joint'].to_dict()
for node,d in personnel_proj_g.nodes(data=True):
    d['joint'] = joint_d.get(node,'')

nx.write_gexf(personnel_proj_g,'personnel_proj.gexf')

personnel_proj_g.number_of_nodes(), personnel_proj_g.number_of_edges()


In [ ]:
all_bio_payee_el_df = yq_total_personnel_filtered.groupby(['BIOGUIDE_ID','PAYEE_CLEANED']).agg({'AMOUNT':'sum'})
all_bio_payee_el_df.reset_index(inplace=True)
all_bio_payee_el_df.columns = ['id','payee','amount']

all_bio_payee_el_df['payee'].value_counts().head(20)

In [ ]:
_df

In [ ]:
top_payees = ['KRYSTAL C KAAI','MICHAEL P DARNER','BRADLEY M BAUMAN','MARIA L LAVERDIERE', 'JOHN D GROM']
_df = yq_total_personnel_filtered.loc[yq_total_personnel_filtered['PAYEE_CLEANED'].isin(top_payees),:]

pd.pivot_table(
    data = _df,
    index = 'YEAR-QUARTER',
    columns = 'PAYEE_CLEANED',
    values = 'AMOUNT',
    aggfunc = 'sum'
).describe()

In [ ]:
all_bio_payee_el_df['payee'].value_counts()

What's up with the Anfinsons?

In [ ]:
_df = yq_total_personnel[yq_total_personnel['PAYEE_CLEANED'].str.contains('ANFINSON')]

pd.pivot_table(
    data = _df,
    index = 'YEAR-QUARTER',
    columns = 'PAYEE_CLEANED',
    values = 'BIOGUIDE_ID',
    aggfunc = 'nunique'
)

In [ ]:
pd.pivot_table(
    data = _df,
    index = 'YEAR-QUARTER',
    columns = 'PAYEE_CLEANED',
    values = 'AMOUNT',
    aggfunc = 'sum'
)

In [ ]:
space_count = yq_total_personnel.loc[:,'PAYEE'].str.split(' ').apply(len)
yq_total_personnel.loc[space_count == 6,'PAYEE_CLEANED']

In [ ]:
post_2016Q4_cleaned_names

In [ ]:
yq_total_personnel.loc[~yq_total_personnel['YEAR-QUARTER'].isin(comma_in_name_quarters),'PAYEE'].apply(name_reorganizer)

In [ ]:
yq_total_personnel.loc[yq_total_personnel['PAYEE'].str.contains('FALKOWSKI'),'YEAR-QUARTER'].values

In [ ]:
yq_total_personnel['PAYEE_CLEANED'].value_counts()

In [ ]:
annual_personnel_to_2016 = annual_office_personnel_df.loc[annual_office_personnel_df['YEAR'] <= 2015,:]
annual_personnel_to_2016[annual_personnel_to_2016.loc[:,'PAYEE'].str.split(',').apply(len) == 1]

#


In [ ]:
annual_office_personnel_df[annual_office_personnel_df['PAYEE'].str.contains('VENDOR')]

## Other data sources

HC: Spending based on ethical rhetoric on social media, public statements, legislation sponsored  
HC: George Santos! Let's look at his spending  
HC: Use House Ethics history to flag members who have previously been subject of an investigation